In [3]:
import pandas as pd
import numpy as np

# Загрузка данных
df = pd.read_csv('DataHomeCredit.csv', usecols=['NAME_EDUCATION_TYPE', 'TARGET'])

In [6]:
df

,TARGET,NAME_EDUCATION_TYPE,edu_natural,edu_calibrated_odds,edu_calibrated
0,0,Higher education,4,-0.353042,2.0
1,0,Secondary / secondary special,2,0.123106,3.0
2,0,Secondary / secondary special,2,0.123106,3.0
3,0,Secondary / secondary special,2,0.123106,3.0
4,0,Higher education,4,-0.353042,2.0
...,...,...,...,...,...
9995,1,Secondary / secondary special,2,0.123106,3.0
9996,1,Secondary / secondary special,2,0.123106,3.0
9997,1,Secondary / secondary special,2,0.123106,3.0
9998,1,Higher education,4,-0.353042,2.0


In [5]:
def goodman_kruskal_gamma(x, y):
    """Вычисление коэффициента корреляции Гамма Гудмэна-Крускала."""
    crosstab = pd.crosstab(x, y)

    ns = 0 # Конкордантные пары
    nd = 0 # Дискордантные пары

    for i in range(crosstab.shape[0]):
        for j in range(crosstab.shape[1]):
            ns += crosstab.iloc[i, j] * crosstab.iloc[(i+1):, (j+1):].sum().sum()
            nd += crosstab.iloc[i, j] * crosstab.iloc[(i+1):, :(j)].sum().sum()

    if (ns + nd) == 0:
        return 0.0
    return (ns - nd) / (ns + nd)

# 1. Естественный порядок категорий
edu_order = {
    'Lower secondary': 1,
    'Secondary / secondary special': 2,
    'Incomplete higher': 3,
    'Higher education': 4,
    'Academic degree': 5
}
df['edu_natural'] = df['NAME_EDUCATION_TYPE'].map(edu_order)

# 2. Калибровка шансов с поправкой Лапласа
alpha = 1
stats = df.groupby('NAME_EDUCATION_TYPE')['TARGET'].agg(['sum', 'count'])

positives = stats['sum'] + alpha
negatives = (stats['count'] - stats['sum']) + alpha
stats['log_odds'] = np.log(positives / negatives)

# Применение калиброванных значений и перевод в ранги
df['edu_calibrated_odds'] = df['NAME_EDUCATION_TYPE'].map(stats['log_odds'])
df['edu_calibrated'] = df['edu_calibrated_odds'].rank(method='dense')

# 3. Расчет и вывод результатов
gamma_natural = goodman_kruskal_gamma(df['edu_natural'], df['TARGET'])
gamma_calibrated = goodman_kruskal_gamma(df['edu_calibrated'], df['TARGET'])

print(f"Гамма (естественный порядок): {gamma_natural:.4f}")
print(f"Гамма (калиброванный порядок): {gamma_calibrated:.4f}")

Гамма (естественный порядок): -0.2034
Гамма (калиброванный порядок): 0.2119
